<a href="https://www.kaggle.com/code/gpreda/meteorites-don-t-fall-where-we-find-them?scriptVersionId=340811891" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## A geographical investigation of discovery bias

A meteorite map looks like a map of where rocks from space hit Earth.

But is it?

This notebook compares meteorites whose falls were **observed** (`Fell`) with meteorites that were **discovered later** (`Found`). If the geography of the two groups differs strongly, the map is not only recording nature — it is also recording where humans can search, preserve, recognize and recover meteorites.

### Questions

1. Do `Fell` and `Found` meteorites have the same geographical distribution?
2. Is there a latitude bias?
3. How exceptional is Antarctica?
4. Did the discovery process change through time?
5. Can we predict whether a meteorite was `Fell` or `Found` using only its metadata and location?
6. Where is the **discovery bias** strongest?

> The goal is not to estimate the true meteorite impact rate. The dataset is a catalogue of known meteorites, so selection and discovery processes are part of the data-generating mechanism.

In [1]:
from pathlib import Path
import glob
import math
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.offline import init_notebook_mode

from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42

# Plotly defaults
PLOTLY_TEMPLATE = "plotly_white"

init_notebook_mode(connected=True)
pio.renderers.default = "notebook_connected"

def show_fig(fig):
    try:
        fig.show(renderer="notebook_connected")
    except Exception:
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
print("Ready.")

Ready.


## 1. Load the dataset


In [2]:
df = pd.read_csv("/kaggle/input/datasets/sujaykapadnis/meteorites-dataset/meteorites.csv")
print(df.columns)
print(df.shape)
df.head()

Index(['name', 'id', 'name_type', 'class', 'mass', 'fall', 'year', 'lat',
       'long', 'geolocation'],
      dtype='object')
(45716, 10)


,name,id,name_type,class,mass,fall,year,lat,long,geolocation
0,Aachen,1,Valid,L5,21.000,Fell,"1,880.000",50.775,6.083,"(50.775, 6.08333)"
1,Aarhus,2,Valid,H6,720.000,Fell,"1,951.000",56.183,10.233,"(56.18333, 10.23333)"
2,Abee,6,Valid,EH4,"107,000.000",Fell,"1,952.000",54.217,-113.000,"(54.21667, -113.0)"
3,Acapulco,10,Valid,Acapulcoite,"1,914.000",Fell,"1,976.000",16.883,-99.900,"(16.88333, -99.9)"
4,Achiras,370,Valid,L6,780.000,Fell,"1,902.000",-33.167,-64.950,"(-33.16667, -64.95)"


## 2. Cleaning

We keep the cleaning deliberately conservative.

Important choices:

- Coordinates must be valid Earth coordinates.
- Years far in the future are removed.
- Zero coordinates are treated cautiously because `(0, 0)` is a common placeholder in geospatial datasets.
- Mass is transformed with `log10(1 + mass)` because meteorite masses are extremely right-skewed.
- `Fell` and `Found` are converted to a binary target.


First, we rename, for convenience, some of the columns:

In [3]:
df.columns = ['name', 'id', 'nametype', 'recclass', 'mass', 'fall', 'year', 
              'latitude', 'longitude', 'geolocation']

In [4]:
d = df.copy()

# Fall/found label
d["fall"] = (
    d["fall"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "fell": "Fell",
        "found": "Found",
        "fall": "Fell",
    })
)

d.loc[~d["fall"].isin(["Fell", "Found"]), "fall"] = np.nan

# Numeric variables
for col in ["mass", "year", "latitude", "longitude"]:
    if col in d.columns:
        d[col] = pd.to_numeric(d[col], errors="coerce")

# Valid coordinate bounds
valid_coord = (
    d["latitude"].between(-90, 90) &
    d["longitude"].between(-180, 180)
)

d = d.loc[valid_coord].copy()

# Remove suspicious exact null-island coordinates.
# Keep rows where only one coordinate is zero.
null_island = (d["latitude"] == 0) & (d["longitude"] == 0)
print("Rows at exact (0,0):", int(null_island.sum()))
d = d.loc[~null_island].copy()

# Conservative year filter.
# Adjust MAX_YEAR if your source contains more recent catalogued falls.
MAX_YEAR = pd.Timestamp.today().year
d.loc[~d["year"].between(860, MAX_YEAR), "year"] = np.nan

if "mass" in d.columns:
    d.loc[d["mass"] < 0, "mass"] = np.nan
    d["log_mass"] = np.log10(1 + d["mass"])

d["is_found"] = (d["fall"] == "Found").astype(float)
d.loc[d["fall"].isna(), "is_found"] = np.nan

d["abs_latitude"] = d["latitude"].abs()

# Useful broad latitude zones
d["latitude_zone"] = pd.cut(
    d["latitude"],
    bins=[-90, -60, -23.5, 23.5, 60, 90],
    labels=[
        "Antarctic",
        "Southern mid-latitudes",
        "Tropics",
        "Northern mid-latitudes",
        "Arctic",
    ],
    include_lowest=True,
)

print("Raw rows:", len(df))
print("Analysis rows:", len(d))
print("\nFall / Found:")
display(d["fall"].value_counts(dropna=False).to_frame("n"))

Rows at exact (0,0): 6214
Raw rows: 45716
Analysis rows: 32186

Fall / Found:


,n
fall,
Found,31090
Fell,1096


### Summary table

In [5]:
summary = pd.DataFrame({
    "metric": [
        "Rows",
        "With valid fall/found label",
        "With mass",
        "With year",
        "Median mass (g)",
        "Median year",
    ],
    "value": [
        len(d),
        d["fall"].notna().sum(),
        d["mass"].notna().sum() if "mass" in d else np.nan,
        d["year"].notna().sum(),
        d["mass"].median() if "mass" in d else np.nan,
        d["year"].median(),
    ]
})

display(summary)

,metric,value
0,Rows,"32,186.000"
1,With valid fall/found label,"32,186.000"
2,With mass,"32,068.000"
3,With year,"32,036.000"
4,Median mass (g),29.900
5,Median year,"1,991.000"


# 3. The map that starts the story

First, plot everything.

At this point, resist explaining the pattern. Let the strange concentrations do the work.


In [6]:
plot_df = d.dropna(subset=["latitude", "longitude"]).copy()

print(plot_df.shape)
hover_cols = [c for c in ["name", "fall", "year", "mass", "recclass"] if c in plot_df.columns]

fig = px.scatter_geo(
    plot_df,
    lat="latitude",
    lon="longitude",
    hover_name="name" if "name" in plot_df.columns else None,
    hover_data=hover_cols,
    opacity=0.55,
    title="Known Meteorites Are Very Unevenly Distributed",
    template=PLOTLY_TEMPLATE,
)

fig.update_traces(marker=dict(size=3))
fig.update_geos(
    projection_type="natural earth",
    showland=True,
    showcountries=True,
    showcoastlines=True,
)
show_fig(fig)

(32186, 14)


Same plot, but with "Found" and "Fell" revealed.

In [7]:
ff = d.dropna(subset=["fall", "latitude", "longitude"]).copy()
print(ff.shape)
fig = px.scatter_geo(
    ff,
    lat="latitude",
    lon="longitude",
    color="fall",
    category_orders={"fall": ["Found", "Fell"]},
    hover_name="name" if "name" in ff.columns else None,
    hover_data=[c for c in ["year", "mass", "recclass"] if c in ff.columns],
    opacity=0.55,
    title="The Crucial Split: Observed Falls vs Later Finds",
    template=PLOTLY_TEMPLATE,
)

fig.update_traces(marker=dict(size=3))
fig.update_geos(
    projection_type="natural earth",
    showland=True,
    showcountries=True,
    showcoastlines=True,
)
show_fig(fig)

(32186, 14)


### First hypothesis

If meteorites fall approximately independently of human search effort, then `Fell` meteorites should be a useful — although imperfect — comparison group.

`Found` meteorites, by contrast, depend heavily on the probability that a meteorite:

1. survives,
2. remains visible,
3. is searched for,
4. is recognized,
5. is collected,
6. enters the catalogue.

That makes the difference between the two groups analytically useful.

# 4. Does latitude matter?

In [8]:
lat_plot = d.dropna(subset=["fall", "latitude"]).copy()

fig = px.histogram(
    lat_plot,
    x="latitude",
    color="fall",
    nbins=72,
    histnorm="probability density",
    barmode="overlay",
    opacity=0.55,
    marginal="box",
    category_orders={"fall": ["Found", "Fell"]},
    title="Fell and Found Meteorites Have Different Latitude Distributions",
    template=PLOTLY_TEMPLATE,
)
show_fig(fig)

In [9]:
LAT_BIN_DEG = 5

lat_binned = d.dropna(subset=["fall", "latitude"]).copy()
lat_binned["lat_bin"] = (
    np.floor(lat_binned["latitude"] / LAT_BIN_DEG) * LAT_BIN_DEG
).astype(int)

lat_stats = (
    lat_binned
    .groupby("lat_bin", observed=True)
    .agg(
        n=("fall", "size"),
        found=("is_found", "sum"),
        found_share=("is_found", "mean"),
    )
    .reset_index()
)

lat_stats["lat_center"] = lat_stats["lat_bin"] + LAT_BIN_DEG / 2

fig = px.scatter(
    lat_stats,
    x="lat_center",
    y="found_share",
    size="n",
    hover_data=["n", "found"],
    title="Where Is a Known Meteorite Most Likely to Be a 'Find'?",
    labels={
        "lat_center": "Latitude",
        "found_share": "Share classified as Found",
    },
    template=PLOTLY_TEMPLATE,
)
fig.add_hline(
    y=d["is_found"].mean(),
    line_dash="dash",
    annotation_text="Global found share"
)
fig.update_yaxes(range=[0, 1.2])
show_fig(fig)

In [10]:
fell_lat = d.loc[d["fall"] == "Fell", "latitude"].dropna()
found_lat = d.loc[d["fall"] == "Found", "latitude"].dropna()

ks = stats.ks_2samp(fell_lat, found_lat)

print(f"Fell n:  {len(fell_lat):,}")
print(f"Found n: {len(found_lat):,}")
print(f"KS statistic: {ks.statistic:.4f}")
print(f"p-value:      {ks.pvalue:.3e}")

print("\nMedian latitude")
print("Fell :", fell_lat.median())
print("Found:", found_lat.median())

print("\nMedian absolute latitude")
print("Fell :", fell_lat.abs().median())
print("Found:", found_lat.abs().median())

Fell n:  1,096
Found n: 31,090
KS statistic: 0.7110
p-value:      0.000e+00

Median latitude
Fell : 36.133335
Found: -72.77361

Median absolute latitude
Fell : 36.21667
Found: 72.77361


The p-value is not the interesting part here — with a catalogue this large, tiny differences can become statistically significant.

The **effect shape** is what matters: *where* do the two distributions diverge?

# 5. The Antarctica reveal

Antarctica is especially valuable for meteorite recovery because dark rocks can be conspicuous against ice and long-lived ice-flow processes can concentrate meteorites in collection zones.

We do not need to assume that explanation yet. First, quantify how strange the catalogue is.


In [11]:
a = d.dropna(subset=["fall", "latitude"]).copy()
a["antarctica"] = a["latitude"] <= -60

antarctic_table = pd.crosstab(
    a["antarctica"].map({True: "Antarctica (<= -60°)", False: "Rest of world"}),
    a["fall"],
)

display(antarctic_table)

antarctic_stats = (
    a.groupby("antarctica")
     .agg(
         n=("fall", "size"),
         found_share=("is_found", "mean"),
         median_mass=("mass", "median") if "mass" in a else ("year", "size"),
     )
)

display(antarctic_stats)

fall,Fell,Found
antarctica,,
Antarctica (<= -60°),0,22099
Rest of world,1096,8991


,n,found_share,median_mass
antarctica,,,
False,10087,0.891,347.000
True,22099,1.000,13.430


How strong is associated Antarctica with "Found"?

In [12]:
ct = pd.crosstab(a["antarctica"], a["fall"])

for col in ["Found", "Fell"]:
    if col not in ct.columns:
        ct[col] = 0

# Haldane-Anscombe correction makes this robust to zero cells
ant_found = ct.loc[True, "Found"] if True in ct.index else 0
ant_fell  = ct.loc[True, "Fell"]  if True in ct.index else 0
rest_found = ct.loc[False, "Found"] if False in ct.index else 0
rest_fell  = ct.loc[False, "Fell"]  if False in ct.index else 0

odds_ratio = (
    (ant_found + 0.5) * (rest_fell + 0.5) /
    ((ant_fell + 0.5) * (rest_found + 0.5))
)
print(f"Antarctica Found/Fell odds ratio: {odds_ratio:,.2f}x")

Antarctica Found/Fell odds ratio: 5,390.00x


Bootstrap confidence interval for Antarctica found-share difference

In [13]:
rng = np.random.default_rng(RANDOM_STATE)

ant = a.loc[a["antarctica"], "is_found"].dropna().to_numpy()
rest = a.loc[~a["antarctica"], "is_found"].dropna().to_numpy()

observed_diff = ant.mean() - rest.mean()

B = 3000
boot = np.empty(B)

for i in range(B):
    ant_sample = rng.choice(ant, size=len(ant), replace=True)
    rest_sample = rng.choice(rest, size=len(rest), replace=True)
    boot[i] = ant_sample.mean() - rest_sample.mean()

ci_low, ci_high = np.quantile(boot, [0.025, 0.975])

print(f"Found-share difference: {observed_diff:.3f}")
print(f"95% bootstrap CI: [{ci_low:.3f}, {ci_high:.3f}]")

Found-share difference: 0.109
95% bootstrap CI: [0.103, 0.115]


# 6. Time: did humans change the map?

In [14]:
yearly = (
    d.dropna(subset=["fall", "year"])
     .assign(year=lambda x: x["year"].astype(int))
     .groupby(["year", "fall"])
     .size()
     .reset_index(name="n")
)

# Focus on modern cataloguing era for readability
YEAR_START = 1800
yearly_modern = yearly.query("year >= @YEAR_START").copy()

fig = px.line(
    yearly_modern,
    x="year",
    y="n",
    color="fall",
    category_orders={"fall": ["Fell", "Found"]},
    title="The Catalogue Changed Dramatically Through Time",
    labels={"n": "Meteorites in catalogue"},
    template=PLOTLY_TEMPLATE,
)
show_fig(fig)

 Decadal view

In [15]:
time_df = d.dropna(subset=["fall", "year"]).copy()
time_df = time_df.query("year >= 1800").copy()
time_df["decade"] = (time_df["year"] // 10 * 10).astype(int)

decadal = (
    time_df.groupby(["decade", "fall"])
           .size()
           .reset_index(name="n")
)

fig = px.bar(
    decadal,
    x="decade",
    y="n",
    color="fall",
    barmode="stack",
    title="Observed Falls and Later Finds by Decade",
    template=PLOTLY_TEMPLATE,
)
show_fig(fig)

Share of records that are "Found" by decade

In [16]:
decade_share = (
    time_df.groupby("decade")
           .agg(
               n=("fall", "size"),
               found_share=("is_found", "mean")
           )
           .reset_index()
)

fig = px.line(
    decade_share,
    x="decade",
    y="found_share",
    markers=True,
    hover_data=["n"],
    title="The Discovery Process Has Changed Over Time",
    labels={"found_share": "Found share"},
    template=PLOTLY_TEMPLATE,
)
fig.update_yaxes(range=[0, 1])
show_fig(fig)

# 7. Mass: do we notice different meteorites in different ways?

In [17]:
if "mass" in d.columns:
    mass_df = d.dropna(subset=["fall", "mass"]).query("mass > 0").copy()

    fig = px.histogram(
        mass_df,
        x="log_mass",
        color="fall",
        nbins=80,
        histnorm="probability density",
        barmode="overlay",
        opacity=0.55,
        category_orders={"fall": ["Found", "Fell"]},
        title="Observed Falls and Finds Have Different Mass Profiles",
        labels={"log_mass": "log10(1 + mass in grams)"},
        template=PLOTLY_TEMPLATE,
    )
    show_fig(fig)

    display(
        mass_df.groupby("fall")["mass"]
               .agg(["count", "median", "mean"])
               .sort_index()
    )

,count,median,mean
fall,,,
Fell,1064,"2,905.000","47,550.754"
Found,30986,27.000,"17,532.804"


# 8. Composition

Meteorite classification can introduce another selection mechanism: some meteorites may be easier to recognize or may be associated with particular search programmes.

Instead of using every fine-grained `recclass`, create a broad family.

In [18]:
def broad_recclass(x):
    if pd.isna(x):
        return "Unknown"

    s = str(x).upper().strip()

    # Coarse scientific grouping for EDA only, not a taxonomic replacement.
    if "IRON" in s:
        return "Iron"
    if "PALLASITE" in s or "MESOSIDERITE" in s:
        return "Stony-iron"
    if s.startswith(("H", "L", "LL", "EH", "EL")):
        return "Chondrite"
    if any(k in s for k in [
        "ACHONDRITE", "EUCRITE", "HOWARDITE", "DIOGENITE",
        "UREILITE", "ANGRITE", "AUBRITE"
    ]):
        return "Achondrite"
    if s.startswith(("CI", "CM", "CO", "CV", "CK", "CR", "CH", "CB")):
        return "Carbonaceous"
    return "Other"


if "recclass" in d.columns:
    d["class_family"] = d["recclass"].map(broad_recclass)

    class_stats = (
        d.dropna(subset=["fall"])
         .groupby("class_family")
         .agg(
             n=("fall", "size"),
             found_share=("is_found", "mean"),
             median_mass=("mass", "median") if "mass" in d else ("fall", "size"),
         )
         .sort_values("n", ascending=False)
    )

    display(class_stats)

    fig = px.bar(
        class_stats.reset_index(),
        x="class_family",
        y="found_share",
        text="n",
        title="Found Share Varies by Broad Meteorite Class",
        labels={"found_share": "Found share", "class_family": "Class family"},
        template=PLOTLY_TEMPLATE,
    )
    fig.update_yaxes(range=[0, 1])
    show_fig(fig)

,n,found_share,median_mass
class_family,,,
Chondrite,28926,0.970,26.920
Iron,962,0.950,"11,310.000"
Carbonaceous,796,0.951,24.050
Achondrite,706,0.915,67.450
Other,608,0.870,6.850
Stony-iron,188,0.941,603.900


# 9. Turn the world into a grid

A point map is visually compelling but difficult to quantify.

We divide Earth into 5° × 5° cells and calculate:

- number of known meteorites,
- number found,
- number observed falling,
- smoothed probability that a catalogue record is `Found`,
- a **Discovery Bias Index**.

The index is descriptive, not causal. It answers:

> *Given that a meteorite is present in the catalogue in this area, how unusually likely is it to be a later discovery rather than an observed fall?*

In [19]:
GRID_DEG = 5

grid = d.dropna(subset=["fall", "latitude", "longitude"]).copy()

grid["lat_cell"] = (
    np.floor(grid["latitude"] / GRID_DEG) * GRID_DEG
).clip(-90, 90 - GRID_DEG)

grid["lon_cell"] = (
    np.floor(grid["longitude"] / GRID_DEG) * GRID_DEG
).clip(-180, 180 - GRID_DEG)

grid_stats = (
    grid.groupby(["lat_cell", "lon_cell"])
        .agg(
            n=("fall", "size"),
            found=("is_found", "sum"),
            raw_found_share=("is_found", "mean"),
        )
        .reset_index()
)

grid_stats["fell"] = grid_stats["n"] - grid_stats["found"]
grid_stats["lat_center"] = grid_stats["lat_cell"] + GRID_DEG / 2
grid_stats["lon_center"] = grid_stats["lon_cell"] + GRID_DEG / 2

display(grid_stats.sort_values("n", ascending=False).head(20))

,lat_cell,lon_cell,n,found,raw_found_share,fell,lat_center,lon_center
23,-80.000,155.000,5193,"5,193.000",1.000,0.000,-77.500,157.500
29,-75.000,35.000,4888,"4,888.000",1.000,0.000,-72.500,37.500
22,-85.000,165.000,3041,"3,041.000",1.000,0.000,-82.500,167.500
21,-85.000,160.000,2536,"2,536.000",1.000,0.000,-82.500,162.500
31,-75.000,75.000,2486,"2,486.000",1.000,0.000,-72.500,77.500
198,15.000,50.000,1541,"1,541.000",1.000,0.000,17.500,52.500
27,-75.000,25.000,1505,"1,505.000",1.000,0.000,-72.500,27.500
249,25.000,15.000,1055,"1,055.000",1.000,0.000,27.500,17.500
7,-90.000,-70.000,857,857.000,1.000,0.000,-87.500,-67.500
228,20.000,55.000,831,831.000,1.000,0.000,22.500,57.500


Bayesian smoothing of found share

In [20]:
# Empirical-Bayes-style beta-binomial smoothing.
# The prior is centered on the global Found share.
global_found_share = grid["is_found"].mean()

PRIOR_STRENGTH = 20
alpha0 = global_found_share * PRIOR_STRENGTH
beta0 = (1 - global_found_share) * PRIOR_STRENGTH

grid_stats["smoothed_found_share"] = (
    grid_stats["found"] + alpha0
) / (
    grid_stats["n"] + alpha0 + beta0
)

# Discovery Bias Index:
# log-odds of smoothed found probability vs global found probability.
eps = 1e-9

def logit(p):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

grid_stats["discovery_bias_index"] = (
    logit(grid_stats["smoothed_found_share"]) -
    logit(global_found_share)
)

print("Global Found share:", round(global_found_share, 4))
display(
    grid_stats.sort_values("discovery_bias_index", ascending=False).head(20)
)

Global Found share: 0.9659


,lat_cell,lon_cell,n,found,raw_found_share,fell,lat_center,lon_center,smoothed_found_share,discovery_bias_index
23,-80.000,155.000,5193,"5,193.000",1.000,0.000,-77.500,157.500,1.000,5.598
29,-75.000,35.000,4888,"4,888.000",1.000,0.000,-72.500,37.500,1.000,5.537
22,-85.000,165.000,3041,"3,041.000",1.000,0.000,-82.500,167.500,1.000,5.065
21,-85.000,160.000,2536,"2,536.000",1.000,0.000,-82.500,162.500,1.000,4.885
31,-75.000,75.000,2486,"2,486.000",1.000,0.000,-72.500,77.500,1.000,4.865
198,15.000,50.000,1541,"1,541.000",1.000,0.000,17.500,52.500,1.000,4.392
27,-75.000,25.000,1505,"1,505.000",1.000,0.000,-72.500,27.500,1.000,4.368
249,25.000,15.000,1055,"1,055.000",1.000,0.000,27.500,17.500,0.999,4.018
7,-90.000,-70.000,857,857.000,1.000,0.000,-87.500,-67.500,0.999,3.815
228,20.000,55.000,831,831.000,1.000,0.000,22.500,57.500,0.999,3.785


Map the discovery-bias index

In [21]:
MIN_CELL_N = 5

bias_map = grid_stats.query("n >= @MIN_CELL_N").copy()

fig = px.scatter_geo(
    bias_map,
    lat="lat_center",
    lon="lon_center",
    color="discovery_bias_index",
    size="n",
    size_max=18,
    hover_data={
        "n": True,
        "found": True,
        "fell": True,
        "smoothed_found_share": ":.3f",
        "discovery_bias_index": ":.3f",
        "lat_center": False,
        "lon_center": False,
    },
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title="Meteorite Discovery Bias Index by 5° × 5° Cell",
    template=PLOTLY_TEMPLATE,
)

fig.update_geos(
    projection_type="natural earth",
    showland=True,
    showcountries=True,
    showcoastlines=True,
)
show_fig(fig)

In [22]:
MIN_CELL_N = 5

bias_map = grid_stats.query("n >= @MIN_CELL_N").copy()

fig = px.scatter_geo(
    bias_map,
    lat="lat_center",
    lon="lon_center",
    color="discovery_bias_index",
    size="n",
    size_max=18,
    hover_data={
        "n": True,
        "found": True,
        "fell": True,
        "smoothed_found_share": ":.3f",
        "discovery_bias_index": ":.3f",
        "lat_center": False,
        "lon_center": False,
    },
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title="Meteorite Discovery Bias Index by 5° × 5° Cell",
    template=PLOTLY_TEMPLATE,
)

fig.update_geos(
    projection_type="natural earth",
    showland=True,
    showcountries=True,
    showcoastlines=True,
)
show_fig(fig)

# ============================================================
# 22. Top spatial anomalies
# ============================================================

top_positive = (
    bias_map.sort_values(
        ["discovery_bias_index", "n"],
        ascending=[False, False]
    )
    .head(15)
    .copy()
)

top_negative = (
    bias_map.sort_values(
        ["discovery_bias_index", "n"],
        ascending=[True, False]
    )
    .head(15)
    .copy()
)

print("Cells unusually dominated by FOUND meteorites")
display(top_positive[
    [
        "lat_center", "lon_center", "n", "found", "fell",
        "smoothed_found_share", "discovery_bias_index"
    ]
])

print("\nCells unusually dominated by observed FALLS")
display(top_negative[
    [
        "lat_center", "lon_center", "n", "found", "fell",
        "smoothed_found_share", "discovery_bias_index"
    ]
])

Cells unusually dominated by FOUND meteorites


,lat_center,lon_center,n,found,fell,smoothed_found_share,discovery_bias_index
23,-77.500,157.500,5193,"5,193.000",0.000,1.000,5.598
29,-72.500,37.500,4888,"4,888.000",0.000,1.000,5.537
22,-82.500,167.500,3041,"3,041.000",0.000,1.000,5.065
21,-82.500,162.500,2536,"2,536.000",0.000,1.000,4.885
31,-72.500,77.500,2486,"2,486.000",0.000,1.000,4.865
198,17.500,52.500,1541,"1,541.000",0.000,1.000,4.392
27,-72.500,27.500,1505,"1,505.000",0.000,1.000,4.368
249,27.500,17.500,1055,"1,055.000",0.000,0.999,4.018
7,-87.500,-67.500,857,857.000,0.000,0.999,3.815
228,22.500,57.500,831,831.000,0.000,0.999,3.785



Cells unusually dominated by observed FALLS


,lat_center,lon_center,n,found,fell,smoothed_found_share,discovery_bias_index
258,27.500,82.500,26,1.000,25.000,0.442,-3.579
257,27.500,77.500,24,1.000,23.000,0.462,-3.498
346,42.500,2.500,21,2.000,19.000,0.520,-3.265
384,47.500,2.500,23,4.000,19.000,0.542,-3.176
330,37.500,137.500,20,3.000,17.000,0.558,-3.112
183,12.500,77.500,16,1.000,15.000,0.564,-3.086
348,42.500,12.500,16,1.000,15.000,0.564,-3.086
385,47.500,7.500,25,7.000,18.000,0.585,-3.002
230,22.500,77.500,14,1.000,13.000,0.598,-2.950
413,52.500,7.500,25,8.000,17.000,0.607,-2.910


# 10. Can location predict whether a meteorite was found?

Now turn the descriptive story into a prediction problem.

This is intentionally **not** a claim that latitude *causes* discovery. Predictability is useful here because if `Fell` vs `Found` can be inferred from geography and catalogue metadata, then the observation process has left a measurable signature in the data.

We use a logistic regression because interpretability matters more than squeezing out every last point of AUC.


In [23]:
model_df = d.dropna(subset=["fall", "latitude", "longitude"]).copy()

model_df["target"] = (model_df["fall"] == "Found").astype(int)

# Geographic features
model_df["abs_latitude"] = model_df["latitude"].abs()

# Longitude is circular: -180° and +180° are adjacent.
lon_rad = np.radians(model_df["longitude"])
lat_rad = np.radians(model_df["latitude"])

model_df["lon_sin"] = np.sin(lon_rad)
model_df["lon_cos"] = np.cos(lon_rad)
model_df["lat_sin"] = np.sin(lat_rad)
model_df["lat_cos"] = np.cos(lat_rad)

# Time
model_df["year_centered"] = model_df["year"] - model_df["year"].median()

# Mass already log-transformed above if available
candidate_numeric = [
    "abs_latitude",
    "lat_sin",
    "lat_cos",
    "lon_sin",
    "lon_cos",
    "year_centered",
]

if "log_mass" in model_df.columns:
    candidate_numeric.append("log_mass")

categorical_features = []
if "class_family" in model_df.columns:
    categorical_features.append("class_family")

numeric_features = [
    c for c in candidate_numeric
    if c in model_df.columns and model_df[c].notna().sum() > 0
]

features = numeric_features + categorical_features

print("Features:", features)
print("Rows:", len(model_df))
print("Target Found share:", model_df["target"].mean())

Features: ['abs_latitude', 'lat_sin', 'lat_cos', 'lon_sin', 'lon_cos', 'year_centered', 'log_mass', 'class_family']
Rows: 32186
Target Found share: 0.9659479276704157


### Logistic regression pipeline

In [24]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

transformers = [("num", numeric_pipe, numeric_features)]

if categorical_features:
    transformers.append(("cat", categorical_pipe, categorical_features))

preprocess = ColumnTransformer(transformers)

model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

pipe = Pipeline([
    ("prep", preprocess),
    ("model", model),
])

X = model_df[features]
y = model_df["target"]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

auc_scores = cross_val_score(
    pipe,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("5-fold ROC AUC:", np.round(auc_scores, 3))
print(f"Mean AUC: {auc_scores.mean():.3f} ± {auc_scores.std():.3f}")

5-fold ROC AUC: [0.976 0.974 0.969 0.975 0.973]
Mean AUC: 0.973 ± 0.002


### Holdout ROC curve

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

pipe.fit(X_train, y_train)

p = pipe.predict_proba(X_test)[:, 1]
pred = (p >= 0.5).astype(int)

auc = roc_auc_score(y_test, p)
fpr, tpr, thresholds = roc_curve(y_test, p)

print(f"Holdout ROC AUC: {auc:.3f}")
print()
print(classification_report(y_test, pred, target_names=["Fell", "Found"]))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fpr,
    y=tpr,
    mode="lines",
    name=f"Logistic regression (AUC={auc:.3f})"
))

fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    name="Random",
    line=dict(dash="dash"),
))

fig.update_layout(
    title="Can We Predict 'Found' vs 'Fell'?",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template=PLOTLY_TEMPLATE,
    width=700,
    height=550,
)

show_fig(fig)

Holdout ROC AUC: 0.971

              precision    recall  f1-score   support

        Fell       0.29      0.90      0.44       274
       Found       1.00      0.92      0.96      7773

    accuracy                           0.92      8047
   macro avg       0.64      0.91      0.70      8047
weighted avg       0.97      0.92      0.94      8047



### Confusion matrix

In [26]:
cm = confusion_matrix(y_test, pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Fell", "Actual Found"],
    columns=["Predicted Fell", "Predicted Found"],
)

display(cm_df)

,Predicted Fell,Predicted Found
Actual Fell,246,28
Actual Found,595,7178


### Permutation importance

In [27]:
perm = permutation_importance(
    pipe,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_sd": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=True)
)

fig = px.bar(
    importance,
    x="importance_mean",
    y="feature",
    orientation="h",
    error_x="importance_sd",
    title="Which Features Help Identify Later Discoveries?",
    labels={
        "importance_mean": "Decrease in ROC AUC when permuted",
        "feature": "",
    },
    template=PLOTLY_TEMPLATE,
)
show_fig(fig)

display(importance.sort_values("importance_mean", ascending=False))

,feature,importance_mean,importance_sd
5,year_centered,0.067,0.002
1,lat_sin,0.036,0.003
6,log_mass,0.026,0.002
0,abs_latitude,0.014,0.002
7,class_family,0.007,0.001
2,lat_cos,0.002,0.001
4,lon_cos,0.001,0.000
3,lon_sin,0.000,0.000


# 11. A harder test: geography alone

If geography alone predicts `Found` vs `Fell`, the observation process has a spatial signature even before using mass, time or meteorite type.


In [28]:
geo_features = [
    "abs_latitude",
    "lat_sin",
    "lat_cos",
    "lon_sin",
    "lon_cos",
]

geo_pipe = Pipeline([
    (
        "prep",
        ColumnTransformer([
            (
                "geo",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]),
                geo_features,
            )
        ])
    ),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    ),
])

geo_auc = cross_val_score(
    geo_pipe,
    model_df[geo_features],
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
)

print("Geography-only AUCs:", np.round(geo_auc, 3))
print(f"Mean geography-only AUC: {geo_auc.mean():.3f}")

Geography-only AUCs: [0.903 0.884 0.892 0.89  0.888]
Mean geography-only AUC: 0.892


# 12. Does the geographical bias change through time?

This is one of the most important robustness checks.

A spatial pattern could be dominated by a single scientific expedition or discovery era. So calculate the latitude profile separately by period.


In [29]:
era_df = d.dropna(subset=["fall", "year", "latitude"]).query("year >= 1800").copy()

era_df["era"] = pd.cut(
    era_df["year"],
    bins=[1799, 1899, 1949, 1979, 1999, np.inf],
    labels=[
        "1800–1899",
        "1900–1949",
        "1950–1979",
        "1980–1999",
        "2000+",
    ]
)

era_df["abs_lat_bin"] = (
    np.floor(era_df["abs_latitude"] / 10) * 10
).astype(int)

era_lat = (
    era_df.groupby(["era", "abs_lat_bin"], observed=True)
          .agg(
              n=("fall", "size"),
              found_share=("is_found", "mean"),
          )
          .reset_index()
)

# Suppress very sparse bins
era_lat = era_lat.query("n >= 5")

fig = px.line(
    era_lat,
    x="abs_lat_bin",
    y="found_share",
    color="era",
    markers=True,
    hover_data=["n"],
    title="The Latitude Bias Is Not Constant Through History",
    labels={
        "abs_lat_bin": "Absolute latitude (10° bins)",
        "found_share": "Found share",
    },
    template=PLOTLY_TEMPLATE,
)
fig.update_yaxes(range=[0, 1])
show_fig(fig)

# 13. Concentration: are finds more geographically clustered?

If organized search programmes and preservation zones matter, `Found` meteorites may be much more concentrated spatially than observed falls.

We measure concentration using the Herfindahl-Hirschman Index (HHI) over equal-angle grid cells.


In [30]:
concentration = grid.copy()
concentration["cell"] = (
    concentration["lat_cell"].astype(str) + "_" +
    concentration["lon_cell"].astype(str)
)

def hhi_for_group(x):
    shares = x.value_counts(normalize=True)
    return np.sum(shares ** 2)

hhi = (
    concentration.groupby("fall")["cell"]
                 .apply(hhi_for_group)
                 .rename("HHI")
                 .to_frame()
)

display(hhi)

if {"Found", "Fell"}.issubset(hhi.index):
    ratio = hhi.loc["Found", "HHI"] / hhi.loc["Fell", "HHI"]
    print(f"Found/Fell spatial concentration ratio: {ratio:.2f}x")

,HHI
fall,
Fell,0.007
Found,0.085


Found/Fell spatial concentration ratio: 11.74x


# 14. One compact "reveal" table

In [31]:
story = []

for group in ["Fell", "Found"]:
    g = d[d["fall"] == group]

    story.append({
        "group": group,
        "n": len(g),
        "median_latitude": g["latitude"].median(),
        "median_abs_latitude": g["abs_latitude"].median(),
        "antarctica_share": (g["latitude"] <= -60).mean(),
        "median_mass_g": g["mass"].median() if "mass" in g else np.nan,
        "median_year": g["year"].median(),
    })

story_df = pd.DataFrame(story).set_index("group")
display(story_df)

,n,median_latitude,median_abs_latitude,antarctica_share,median_mass_g,median_year
group,,,,,,
Fell,1096,36.133,36.217,0.000,"2,905.000","1,924.000"
Found,31090,-72.774,72.774,0.711,27.000,"1,992.000"


# 15. Sanity checks

A few things can easily produce misleading results in this dataset:

- duplicate records,
- extreme masses,
- missing years,
- relict/non-valid records,
- coordinates with suspicious placeholders.

In [32]:
check_cols = [
    c for c in ["name", "year", "latitude", "longitude", "mass"]
    if c in d.columns
]

if check_cols:
    duplicate_mask = d.duplicated(check_cols, keep=False)
    print("Potential duplicate rows:", int(duplicate_mask.sum()))
    display(d.loc[duplicate_mask, check_cols + ["fall"]].head(20))

if "nametype" in d.columns:
    print("\nNameType distribution:")
    display(d["nametype"].value_counts(dropna=False).head(20))

Potential duplicate rows: 0


,name,year,latitude,longitude,mass,fall



NameType distribution:


nametype
Valid     32116
Relict       70
Name: count, dtype: int64

In [33]:
if "nametype" in d.columns:
    valid_mask = (
        d["nametype"]
        .astype(str)
        .str.lower()
        .str.strip()
        .eq("valid")
    )

    valid_d = d.loc[valid_mask].copy()

    print("Valid subset rows:", len(valid_d))
    print(
        valid_d.groupby("fall")["abs_latitude"]
               .agg(["count", "median", "mean"])
    )

    valid_ant = (
        valid_d.assign(antarctica=valid_d["latitude"] <= -60)
               .groupby("antarctica")["is_found"]
               .agg(["count", "mean"])
    )

    print("\nAntarctica sensitivity check:")
    display(valid_ant)
else:
    print("No 'nametype' column; skipping.")

Valid subset rows: 32116
       count  median   mean
fall                       
Fell    1096  36.217 35.156
Found  31020  72.774 62.629

Antarctica sensitivity check:


,count,mean
antarctica,,
False,10020,0.891
True,22096,1.000


# 16. Optional robustness check: remove Antarctica

Antarctica is such a strong discovery environment that it can dominate the result.

If `Found` and `Fell` still differ substantially after Antarctica is removed, the discovery-bias story is broader than a single region.

In [34]:
no_ant = d.loc[d["latitude"] > -60].dropna(subset=["fall", "latitude"]).copy()

fell_no_ant = no_ant.loc[no_ant["fall"] == "Fell", "latitude"]
found_no_ant = no_ant.loc[no_ant["fall"] == "Found", "latitude"]

ks_no_ant = stats.ks_2samp(fell_no_ant, found_no_ant)

print("Without Antarctica:")
print(f"KS statistic: {ks_no_ant.statistic:.4f}")
print(f"p-value:      {ks_no_ant.pvalue:.3e}")
print()
print("Median absolute latitude:")
print("Fell :", fell_no_ant.abs().median())
print("Found:", found_no_ant.abs().median())

Without Antarctica:
KS statistic: 0.4115
p-value:      1.585e-149

Median absolute latitude:
Fell : 36.21667
Found: 27.06967


In [35]:
fig = px.histogram(
    no_ant,
    x="latitude",
    color="fall",
    nbins=60,
    histnorm="probability density",
    barmode="overlay",
    opacity=0.55,
    category_orders={"fall": ["Found", "Fell"]},
    title="The Bias Beyond Antarctica",
    template=PLOTLY_TEMPLATE,
)
show_fig(fig)

# 17. Candidate conclusion generator

This cell prints numerical facts only. Use them to write the final prose after inspecting all plots.

Avoid language such as *"meteorites fall more often in..."* unless you have independent exposure data. The catalogue measures **known meteorites**, not the true impact process.


In [36]:
facts = {}

facts["total_records_used"] = len(d)
facts["found_n"] = int((d["fall"] == "Found").sum())
facts["fell_n"] = int((d["fall"] == "Fell").sum())

facts["global_found_share"] = float(d["is_found"].mean())

facts["antarctica_records"] = int(
    (d["latitude"] <= -60).sum()
)

facts["antarctica_found_share"] = float(
    d.loc[d["latitude"] <= -60, "is_found"].mean()
)

facts["rest_found_share"] = float(
    d.loc[d["latitude"] > -60, "is_found"].mean()
)

facts["latitude_ks_statistic"] = float(ks.statistic)
facts["latitude_ks_pvalue"] = float(ks.pvalue)

facts["geo_only_cv_auc_mean"] = float(geo_auc.mean())
facts["full_model_cv_auc_mean"] = float(auc_scores.mean())

for k, v in facts.items():
    if isinstance(v, float):
        print(f"{k:32s}: {v:.4f}")
    else:
        print(f"{k:32s}: {v:,}")

total_records_used              : 32,186
found_n                         : 31,090
fell_n                          : 1,096
global_found_share              : 0.9659
antarctica_records              : 22,099
antarctica_found_share          : 1.0000
rest_found_share                : 0.8913
latitude_ks_statistic           : 0.7110
latitude_ks_pvalue              : 0.0000
geo_only_cv_auc_mean            : 0.8916
full_model_cv_auc_mean          : 0.9732


# Conclusion

A map of known meteorites is **not simply a map of meteorite impacts**.

The crucial distinction is between meteorites whose falls were observed and meteorites discovered later. If those two groups have very different spatial distributions, then geography is encoding the human discovery process as well as the natural phenomenon.

The strongest interpretation should be based on the notebook's actual results, but the evidence to look for is:

- extreme concentrations of `Found` meteorites in specific regions,
- a different latitude distribution for `Found` and `Fell`,
- unusually high `Found` share in Antarctica,
- changes in the catalogue through historical time,
- measurable predictive power from geography alone,
- persistence of the effect even after removing Antarctica.

That leads to the central idea:

> **We began by mapping rocks from space. We ended up mapping the places where humans are unusually good at finding them.**
